# imports

In [17]:
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, accuracy_score
import mlflow
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import mlflow.tensorflow
from tensorflow.keras.callbacks import EarlyStopping

# config

In [18]:
ASSET = "BTC"
INTERVAL = "1h"

In [19]:
TIMESTEPS = 24

# mlflow

In [20]:
mlflow.set_tracking_uri("http://localhost:5000")

experiment_name = f"{ASSET}_{INTERVAL}_deep_learning"
mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='/mlruns/13', creation_time=1778414378831, experiment_id='13', last_update_time=1778414378831, lifecycle_stage='active', name='BTC_1h_deep_learning', tags={}, trace_location=None, workspace='default'>

# load data

In [21]:
train_df = pd.read_parquet('../../../data/processed/train_btc_1h.parquet')
test_df = pd.read_parquet('../../../data/processed/test_btc_1h.parquet')

# splitting

In [22]:
features = [col for col in train_df.columns if col not in ['target_1h', 'target_direction']]
X_train = train_df[features]
y_train = train_df['target_direction']
X_test = test_df[features]
y_test = test_df['target_direction']

# checking the train data if scaled or not

In [23]:
print(X_train.describe().loc[['mean', 'std']].T.head(5))

                          mean       std
volume            4.386989e-17  1.000012
daily_volatility  6.381076e-17  1.000012
rsi_14            5.919112e-16  1.000012
macd              1.994086e-18  1.000012
macd_signal       1.329391e-18  1.000012


# LSTM

In [24]:
def create_sequences(X, y, time_steps=24):
    """
    Transforms 2D tabular data into 3D tensors for LSTM consumption.
    X: Numpy array of features
    y: Target series
    time_steps: historical periods to look back
    """
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:(i + time_steps)])
        ys.append(y.iloc[i + time_steps])
        
    return np.array(Xs), np.array(ys)


X_train_seq, y_train_seq = create_sequences(X_train.values, y_train, TIMESTEPS)
X_test_seq, y_test_seq = create_sequences(X_test.values, y_test, TIMESTEPS)

print(f"Training Shape (Samples, TimeSteps, Features): {X_train_seq.shape}")
print(f"Testing Shape: {X_test_seq.shape}")

Training Shape (Samples, TimeSteps, Features): (42735, 24, 37)
Testing Shape: (10666, 24, 37)


# model

In [25]:
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(TIMESTEPS, X_train_seq.shape[2])),
    Dropout(0.2),
    
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    
    Dense(16, activation='relu'),
    
    Dense(1, activation='sigmoid')
])

c:\Users\Manindra\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


# compile model

In [26]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [27]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 24, 64)         │        26,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 39,073 (152.63 KB)

 Trainable params: 39,073 (152.63 KB)

 Non-trainable params: 0 (0.00 B)

# mlflow tensorflow 

In [28]:
mlflow.tensorflow.autolog()

2026/05/10 14:42:00 WARNING mlflow.utils.autologging_utils: MLflow tensorflow autologging is known to be compatible with 2.16.2 <= tensorflow, but the installed version is 2.16.1. If you encounter errors during autologging, try upgrading / downgrading tensorflow to a compatible version, or try upgrading MLflow.


# early stopping

In [29]:
early_stopping = EarlyStopping(
    monitor='val_loss', 
    patience=5,
    restore_best_weights=True
)

# run, fit model with mlflow

In [30]:
with mlflow.start_run(run_name="LSTM_Baseline_run3"):
    
    mlflow.log_param("time_steps", TIMESTEPS)
    mlflow.log_param("model_type", "LSTM_2_Layer")
    
    print("started training")
    
    history = model.fit(
        X_train_seq, y_train_seq,
        epochs=50,
        batch_size=64,
        validation_split=0.2,
        callbacks=[early_stopping],
        verbose=1
    )
    
    test_loss, test_acc = model.evaluate(X_test_seq, y_test_seq, verbose=0)
    
    mlflow.log_metric("test_loss", test_loss)
    mlflow.log_metric("test_accuracy", test_acc)
    mlflow.keras.log_model(model, "lstm_model")
    
    print(f"\n--- Training Complete ---")
    print(f"Test Accuracy: {test_acc:.4f}")

started training


Epoch 1/50
533/535 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5064 - loss: 0.6957

535/535 ━━━━━━━━━━━━━━━━━━━━ 12s 16ms/step - accuracy: 0.5064 - loss: 0.6956 - val_accuracy: 0.5019 - val_loss: 0.6959
Epoch 2/50
531/535 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5090 - loss: 0.6935

535/535 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.5090 - loss: 0.6935 - val_accuracy: 0.5050 - val_loss: 0.6939
Epoch 3/50
535/535 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.5210 - loss: 0.6921 - val_accuracy: 0.5136 - val_loss: 0.6941
Epoch 4/50
534/535 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5228 - loss: 0.6918

535/535 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.5228 - loss: 0.6918 - val_accuracy: 0.5119 - val_loss: 0.6936
Epoch 5/50
533/535 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5297 - loss: 0.6909

535/535 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.5296 - loss: 0.6909 - val_accuracy: 0.5188 - val_loss: 0.6929
Epoch 6/50
535/535 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.5257 - loss: 0.6909 - val_accuracy: 0.5197 - val_loss: 0.6932
Epoch 7/50
535/535 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.5365 - loss: 0.6895 - val_accuracy: 0.5134 - val_loss: 0.6948
Epoch 8/50
535/535 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.5386 - loss: 0.6892 - val_accuracy: 0.5137 - val_loss: 0.6946
Epoch 9/50
535/535 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.5326 - loss: 0.6901 - val_accuracy: 0.5128 - val_loss: 0.7015
Epoch 10/50
535/535 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.5414 - loss: 0.6887 - val_accuracy: 0.5157 - val_loss: 0.6988
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 284ms/step


2026/05/10 14:43:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/10 14:43:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/10 14:43:48 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.



--- Training Complete ---
Test Accuracy: 0.5085
🏃 View run LSTM_Baseline_run3 at: http://localhost:5000/#/experiments/13/runs/395b48652f5b49408525b8987639fcea
🧪 View experiment at: http://localhost:5000/#/experiments/13
